# Carga de datos y librerías

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/datos_modelo.csv")

# Feature Engineering

## One-hot encoding

In [ ]:
df.columns

In [ ]:
columnas = ['junction', 'tunnel', 'access', 'bridge']
df = df.drop(columns=columnas)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
dummies = pd.get_dummies(df['highway'], prefix='highway', drop_first=False)
df_encoded = pd.concat([df, dummies], axis=1)


In [ ]:
df_encoded.columns

## Tipo de datos

In [ ]:
df_encoded.dtypes

Solo tenemos que cambiar el tipo de datos de una columna

In [ ]:
df_encoded['vehiculos_por_hora'] = pd.to_numeric(df_encoded['vehiculos_por_hora'], errors='coerce')

## Creación variable categórica (variable de salida)

Dado que el objetivo del modelo es demostrar la viabilidad de predecir la intensidad del tráfico a partir de atributos estáticos y observaciones limitadas en un instante del tiempo, clasificarla en categorías (baja, media, alta, pico) simplifica el problema y reduce la sensibilidad al ruido o valores atípicos típicos de las espiras. Este enfoque permite evaluar de forma más robusta la capacidad del modelo para reconocer patrones espaciales de congestión sin requerir series temporales extensas ni una calibración fina de valores continuos de flujo. Además, facilita la interpretación visual de los resultados en mapas de congestión y escenarios simulados.

Creamos el conjunto de para el modelo (los que no tienen nulos en la salida)

In [ ]:
df_modelo = df_encoded.dropna(subset=['vehiculos_por_hora'])

In [ ]:
df_modelo['nivel_intensidad'] = pd.qcut(df_modelo['vehiculos_por_hora'], q=4, labels=['bajo', 'medio', 'alto', 'pico'])

In [ ]:
df_modelo[['vehiculos_por_hora', 'nivel_intensidad']].head(10)

In [ ]:
df_encoded.dtypes

In [ ]:
df_encoded["geometry"] = df_encoded["geometry"].apply(wkt.loads)

# AED

Observemos el mapa con la variable salida coloreada dependiendo de su valor

In [ ]:
# --- Crear GeoDataFrame de espiras ---
gdf_espiras = gpd.GeoDataFrame(
    df_encoded,  # DataFrame con 'latitud' y 'longitud'
    geometry=gpd.points_from_xy(df.longitud, df.latitud),
    crs="EPSG:4326"
)

# --- Colores por nivel de intensidad ---
colores = {
    'bajo': '#FFF176',
    'medio': '#FFA726',
    'alto': '#EF5350',
    'pico': '#B71C1C',
    'sin_dato': '#BDBDBD'
}

m = folium.Map(location=[39.4699, -0.3763], zoom_start=13, tiles="cartodb positron")

# --- Diccionario de tramos con intensidad ---
niveles_dict = df_modelo.set_index('identificador')['nivel_intensidad'].to_dict()

# --- Añadir tramos coloreados ---
for _, row in df_encoded.iterrows():
    tramo_id = row['identificador']
    nivel = niveles_dict.get(tramo_id, 'sin_dato')
    color = colores.get(nivel, '#BDBDBD')

    if row.geometry.geom_type == 'LineString':
        coords = [(lat, lon) for lon, lat in row.geometry.coords]
        folium.PolyLine(
            coords,
            color=color,
            weight=4,
            opacity=0.8,
            popup=f"ID: {tramo_id}<br>Nivel: {nivel}"
        ).add_to(m)

# --- Añadir puntos de espiras (con control de NaN) ---
for _, row in gdf_espiras.iterrows():
    # Comprobamos que la geometría no sea nula y tenga coordenadas válidas
    if row.geometry is not None and not row.geometry.is_empty:
        lat, lon = row.geometry.y, row.geometry.x
        
        # Evita puntos con coordenadas NaN
        if pd.notna(lat) and pd.notna(lon):
            folium.CircleMarker(
                location=[lat, lon],
                radius=5,
                color="#1565C0",
                fill=True,
                fill_opacity=0.8,
                popup=f"Espira ID: {row.get('espira_id', 'N/A')}"
            ).add_to(m)
        else:
            # Opcional: mensaje de control o log de advertencia
            # print(f"Coordenadas NaN para espira ID {row.get('espira_id', 'N/A')}")
            pass
    else:
        # Geometría vacía o inválida, se ignora
        # print(f"Geometría no válida para espira ID {row.get('espira_id', 'N/A')}")
        pass



m


Conociendo Valencia parece ser que los datos tienen sentido

Veamos de forma estadística de que forma se pueden relacionar la intensidad con otras variables

In [ ]:
df_modelo.columns

Visualicemos la tabla de contingencia de la variable de salida (nivel_intensidad) con el tipo de vía

In [ ]:
# Tabla de contingencia normalizada (por filas)
tabla_pct = pd.crosstab(df_modelo['nivel_intensidad'],
                        df_modelo['highway'],
                        normalize='index')

# Creamos una matriz de etiquetas solo para valores > 0.10 (10%)
annot = np.where(tabla_pct > 0.10,
                 (tabla_pct * 100).round(1).astype(str) + '%',
                 '')

# Graficamos
plt.figure(figsize=(8,5))
sns.heatmap(tabla_pct,
            cmap='YlOrRd',
            annot=annot,      # usamos nuestras etiquetas personalizadas
            fmt='',
            cbar_kws={'label': 'Proporción'})
plt.title("Distribución de tipo de vía por nivel de intensidad (>10%)")
plt.xlabel("Tipo de vía")
plt.ylabel("Nivel de intensidad")
plt.show()

Se observa claramente como los niveles de intensidad medio y bajo, se relacionan más con calles residenciales o terciarias. Por otra parte los valores altos (alto y pico) se relacionan más con tipos de vía primaria y secundaria. Este hallazgo en la tabla de contingencia confirma la hipótesis observada en el mapa, lo que nos muestra consistencia lógica en los datos.

Veamos si la distribución de la velocidad máxima permitida difiere entre los diferentes niveles de intensidad

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

colores = {
    'bajo': "#FDCF00", 
    'medio': "#B46C00",
    'alto': '#EF5350', 
    'pico': '#B71C1C'  
}

for nivel, color in colores.items():
    subset = df_modelo[df_modelo['nivel_intensidad'] == nivel]
    if not subset.empty:
        sns.kdeplot(
            subset['maxspeed'],
            label=nivel.capitalize(),
            fill=True,
            alpha=0.4,
            color=color
        )

plt.title("Distribución de 'maxspeed' por nivel de intensidad")
plt.xlabel("Velocidad máxima (km/h)")
plt.ylabel("Densidad")
plt.legend(title="Nivel de intensidad")
plt.grid(alpha=0.3)
plt.show()


Parece que hay una clara diferencia entre las distribuciones, veámoslo estadísticamente:

“¿Las velocidades máximas (maxspeed) son significativamente diferentes entre los distintos niveles de intensidad de tráfico?”

H₀ (hipótesis nula): todas las medias de maxspeed son iguales entre los grupos de intensidad.

H₁ (hipótesis alternativa): al menos una media difiere significativamente.

In [ ]:
from scipy.stats import f_oneway

# Agrupar valores por nivel de intensidad
grupos = [df_modelo.loc[df_modelo['nivel_intensidad'] == nivel, 'maxspeed'].dropna()
          for nivel in df_modelo['nivel_intensidad'].unique()]

# ANOVA de un factor
f_stat, p_val = f_oneway(*grupos)

print(f"F = {f_stat:.3f}, p-valor = {p_val:.4f}")


El p-valor < 0.05, rechazamos H0, hay diferencias significativas entre los grupos de intensidad, esto es una muestra de que maxspeed será una variable discriminatoria en nuestro modelo.

In [ ]:
# Asegurar que no haya NaN ni inf
df_tukey = df_modelo[['maxspeed', 'nivel_intensidad']].dropna()
df_tukey = df_tukey[np.isfinite(df_tukey['maxspeed'])]

# Comprobar tamaño por grupo
print(df_tukey['nivel_intensidad'].value_counts())


In [ ]:
df_tukey['nivel_intensidad'] = df_tukey['nivel_intensidad'].astype('category')


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df_tukey['maxspeed'],
                          groups=df_tukey['nivel_intensidad'],
                          alpha=0.05)
print(tukey)


Comprobamos mediante el test de Tukey que sí hay diferencias significativas prácticamente entre casi todas las clases

# Modelado Predictivo

Objetivo: Predecir el nivel de intensidad (vehículos_por_hora categorizado) en tramos sin sensores (espiras)

¿Cómo puede ayudar?

Aunque este modelo se desarolla sobre una instancia temporal concreta, su objetivo es demostrar la viabilidad de un sistema que, en el futuro, podría procesar datos históricos y en tiempo real (por horas, días o años).
Este enfoque permitiría construir una plataforma interactiva donde el Ayuntamiento de Paiporta pueda visualizar, analizar y simular escenarios de movilidad urbana.

A través de filtros por año, día u hora, se podrían identificar patrones de congestión, analizar qué días o zonas presentan mayor colapso, y tomar decisiones informadas —por ejemplo, ajustar la frecuencia de autobuses, planificar nuevas paradas o rediseñar tramos viales—.

En definitiva, se trata de un prototipo de apoyo a la planificación urbana, orientado a convertir los datos de movilidad en decisiones operativas y estratégicas.

## Selección de target y variables predictivas

In [ ]:
df_modelo.columns

In [ ]:
columnas_eliminar = [
    "osmid", "id_tramo", "name", "highway", "maxspeed", 
    "geometry", "punto_A", "punto_B", "lon_A", "lat_A", "lon_B", "lat_B", "id_punto_medida", "vehiculos_por_hora",
    "coordenadas", "longitud", "latitud", "espira_id", "identificador", "angulo_sentido_circulacion"
    ]

df_modelo = df_modelo.drop(columns= columnas_eliminar)

In [ ]:
df_modelo.columns

## Split Train/Test

In [ ]:
df = df_modelo.copy()
X = df.drop(columns=['nivel_intensidad'])
y = df['nivel_intensidad']

# Split 70/30 con semilla fija para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,          # 30% para test
    random_state=42,        # fija la aleatoriedad
    stratify=y              # mantiene proporciones 
)

print(f"Tamaño total: {len(df)}")
print(f"Entrenamiento: {len(X_train)} muestras ({len(X_train)/len(df):.1%})")
print(f"Test: {len(X_test)} muestras ({len(X_test)/len(df):.1%})")


## Modelos de referencia inicial

### Naive 

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Crear el modelo naive
naive_model = DummyClassifier(strategy="most_frequent")

# Entrenarlo sobre el conjunto de entrenamiento
naive_model.fit(X_train, y_train)

# Predecir sobre el conjunto de test
y_pred_naive = naive_model.predict(X_test)

# Evaluar desempeño
acc = accuracy_score(y_test, y_pred_naive)
print(f"Accuracy modelo Naïve: {acc:.3f}\n")

print("Classification report:")
print(classification_report(y_test, y_pred_naive))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_naive))


El modelo siempre predice la clase medio, dando así un recall de 1 para dicho grupo. Además muestra un accuracy de 0.255, lo cual es un valor muy bajo

### Árbol de decisión

Inicialmente planteamos utilizar una regresión logística como modelo de referencia (baseline). Sin embargo, este tipo de modelo asume independencia entre las observaciones, una condición que no se cumple plenamente en nuestro caso, ya que las mediciones de las espiras están espacial y posiblemente temporalmente correlacionadas.
Por este motivo, optamos por emplear un árbol de decisión básico, que no requiere dicha independencia estricta y puede capturar relaciones no lineales entre las variables de manera más flexible.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

clases = np.unique(y_train)
pesos_auto = compute_class_weight(class_weight="balanced", classes=clases, y=y_train)
pesos_dict = dict(zip(clases, pesos_auto))

print("Pesos calculados automáticamente:", pesos_dict)

Podemos observar que los pesos están balanceados

Entrenemos el modelo

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)

# === Modelo base ===
tree_model = DecisionTreeClassifier(
    criterion='gini',       # o 'entropy'
    splitter='random',        # o 'random'
    max_depth=15,         # controla profundidad del árbol
    min_samples_split=5,    # mínimo de muestras para dividir un nodo
    min_samples_leaf=1,     # mínimo de muestras en una hoja
    max_features=None,      # nº de features consideradas por split
    class_weight=None,      # ponderación de clases (útil si están desbalanceadas)
    random_state=42         # reproducibilidad
)

# Entrenar
tree_model.fit(X_train, y_train)

# Predecir
y_pred = tree_model.predict(X_test)

# Evaluar
print("Accuracy:", tree_model.score(X_test, y_test))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nKappa:", cohen_kappa_score(y_test, y_pred))

Da un Kappa bajo, lo cuál indica que hace mejores predicciones que el azar, pero aún no muy buenas. Cabe destacar también que el acuraccy ha aumentado.

### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    balanced_accuracy_score
)

rf_model = RandomForestClassifier(
    n_estimators=300,         # número de árboles (100-500 suele ir bien)
    criterion="gini",         # "gini" o "entropy"
    max_depth=None,           # profundidad máxima (None = hasta que cada hoja sea pura)
    min_samples_split=2,      # mínimo de muestras para dividir un nodo
    min_samples_leaf=1,       # mínimo de muestras por hoja
    max_features="sqrt",      # nº de features consideradas en cada split ("sqrt" o "log2" para clasificación)
    class_weight="balanced",  # ajusta peso de clases automáticamente
    random_state=42,          # reproducibilidad
    n_jobs=-1,                # usa todos los núcleos del procesador (más rápido)
    oob_score=True            # usa muestras fuera de bolsa para validación interna
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", rf_model.score(X_test, y_test))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

import pandas as pd
import matplotlib.pyplot as plt

feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(feature_importances['Feature'][:15][::-1], feature_importances['Importance'][:15][::-1])
plt.title("Top 15 Features más importantes (Random Forest)")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

print("\n Top 10 variables que más explican la congestión:")
print(feature_importances.head(10))


Sorprendentemente las variables estáticas que más explican la congestión en el estudio de google son:
* length
* lanes
* width
* speed_limit

Observamos que aquí también es el caso. 

Cabe mencionar que en el paper de Google "Scalable Learning of Segment-Level Traffic Congestion Functions", ayuda mucho los datos históricos a predecir, cosa que no implementamos nosotros (de momento).

## Modelo principal

## MLP

In [ ]:
df_modelo.columns

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# === 1️⃣ Definir columnas ===
num_cols = ['lanes', 'maxspeed_final', 'length']

# El resto son dummy, excluyendo el target
dummy_cols = [c for c in X_train.columns if c not in num_cols + ['nivel_intensidad']]

# === 2️⃣ Escalado solo a numéricas ===
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[num_cols])
X_test_scaled_num = scaler.transform(X_test[num_cols])

# === 3️⃣ Convertir todas las columnas dummy a float (importante)
X_train[dummy_cols] = X_train[dummy_cols].astype(float)
X_test[dummy_cols] = X_test[dummy_cols].astype(float)

# === 4️⃣ Concatenar numéricas escaladas + dummies intactas ===
X_train_scaled = np.concatenate([X_train_scaled_num, X_train[dummy_cols].values], axis=1)
X_test_scaled = np.concatenate([X_test_scaled_num, X_test[dummy_cols].values], axis=1)

# === 5️⃣ Comprobación rápida ===
print("✅ Escalado correcto:")
print(f"Numéricas escaladas: {num_cols}")
print(f"Dummies (sin nivel_intensidad): {len(dummy_cols)} columnas -> {dummy_cols[:5]} ...")
print(f"Shape X_train_scaled: {X_train_scaled.shape}")


In [ ]:
# 1.1 Asegurar que son arrays numéricos 2D
assert isinstance(X_train_scaled, np.ndarray) and X_train_scaled.ndim == 2
assert isinstance(X_test_scaled,  np.ndarray) and X_test_scaled.ndim == 2

# 1.2 Misma cantidad de columnas en train y test
assert X_train_scaled.shape[1] == X_test_scaled.shape[1], "Train/Test con distinto nº de features"

# 1.3 Sin NaN/Inf
def _check_finite(name, X):
    if not np.isfinite(X).all():
        bad = np.argwhere(~np.isfinite(X))
        raise ValueError(f"{name} contiene NaN/Inf en {bad[:5]} (muestra)")
_check_finite("X_train_scaled", X_train_scaled)
_check_finite("X_test_scaled",  X_test_scaled)

# 1.4 Sin columnas constantes (varianza cero)
var_train = X_train_scaled.var(axis=0)
cols_const = np.where(var_train == 0)[0]

if len(cols_const) > 0:
    print("Eliminando columnas constantes:", [dummy_cols[i - len(num_cols)] if i >= len(num_cols) else num_cols[i] for i in cols_const])
    X_train_scaled = np.delete(X_train_scaled, cols_const, axis=1)
    X_test_scaled = np.delete(X_test_scaled, cols_const, axis=1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print("Clases codificadas:", list(le.classes_))
print("Ejemplo y_train:", y_train[:10])

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# === 1️⃣ Definir el modelo base ===
base_mlp = MLPClassifier(
    activation='relu',
    solver='adam',
    early_stopping=True,
    random_state=42,
    max_iter=500
)

# === 2️⃣ Definir la malla de hiperparámetros (grid) ===
param_grid = {
    # Tamaño y número de capas ocultas
    'hidden_layer_sizes': [
        (16,),         # 1 capa pequeña
        (32, 16),      # 2 capas medianas
        (64, 32, 16)   # 3 capas más profunda
    ],
    # Regularización (alpha = weight decay)
    'alpha': [0.0001, 0.001, 0.01],
    # Tasa de aprendizaje inicial
    'learning_rate_init': [0.001, 0.0005],
    # Batch size (tamaño de lote)
    'batch_size': [32, 64]
}

# === 3️⃣ Configurar búsqueda con validación cruzada ===
grid_mlp = GridSearchCV(
    estimator=base_mlp,
    param_grid=param_grid,
    scoring='f1_macro',     # métrica objetivo (puedes cambiar por 'balanced_accuracy')
    cv=5,                   # 5 folds de validación cruzada
    n_jobs=-1,              # usa todos los núcleos disponibles
    verbose=2
)

# === 4️⃣ Ejecutar búsqueda ===
print("🔍 Buscando la mejor combinación de hiperparámetros...")
grid_mlp.fit(X_train_scaled, y_train)

# === 5️⃣ Resultados ===
print("\n✅ Mejor combinación encontrada:")
print(grid_mlp.best_params_)
print(f"✅ Mejor F1_macro (CV): {grid_mlp.best_score_:.3f}")

# === 6️⃣ Evaluar en el conjunto de test ===
best_mlp = grid_mlp.best_estimator_
y_pred = best_mlp.predict(X_test_scaled)

print("\n📊 Evaluación en test:")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
print(f"F1_macro: {f1_score(y_test, y_pred, average='macro'):.3f}")
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))



A pesar de haber usado un grid search, el MLP no es buena opción por el número reducido de datos que tenemos. Es por ello que muestra métricas muy pobres como un Kappa de 0.381, o un Balanced Accuracy de 0.536.

## XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_tuned = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    learning_rate=0.03,
    n_estimators=600,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=0.5,
    random_state=42,
    n_jobs=-1
)

evals = [(X_train_scaled, y_train), (X_test_scaled, y_test)]
xgb_tuned.fit(
    X_train_scaled, y_train,
    eval_set=evals,
    verbose=False
)


El modelo de Gradient Boosted Trees (XGBoost) alcanzó un rendimiento superior al del Multilayer Perceptron, con un F1-macro de 0.55, un Kappa de 0.42 y una Balanced Accuracy de 0.56.
Estos valores reflejan una capacidad predictiva moderada en la clasificación de los niveles de intensidad, suficiente para validar la viabilidad del enfoque.
El modelo demuestra que, incluso sin datos de flujo temporal, las variables geométricas y funcionales de la red vial contienen información suficiente para estimar patrones de congestión.

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score

# Métricas en entrenamiento
y_pred_train = xgb_tuned.predict(X_train_scaled)

print("🔹 TRAIN:")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_train, y_pred_train):.3f}")
print(f"F1_macro: {f1_score(y_train, y_pred_train, average='macro'):.3f}")
print(f"Kappa: {cohen_kappa_score(y_train, y_pred_train):.3f}")

# Métricas en test
y_pred_test = xgb_tuned.predict(X_test_scaled)

print("\n🔹 TEST:")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_test):.3f}")
print(f"F1_macro: {f1_score(y_test, y_pred_test, average='macro'):.3f}")
print(f"Kappa: {cohen_kappa_score(y_test, y_pred_test):.3f}")


In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, test_scores = learning_curve(
    xgb_tuned, X_train_scaled, y_train,
    cv=5, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_mean, label='Train F1', marker='o')
plt.plot(train_sizes, test_mean, label='Validation F1', marker='o')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('F1_macro')
plt.title('Curvas de aprendizaje - XGBoost')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(xgb_tuned, X_train_scaled, y_train, cv=5, scoring='f1_macro')
print(f"Scores por fold: {cv_scores}")
print(f"Media: {cv_scores.mean():.3f}, Desviación: {cv_scores.std():.3f}")
